# Chapter 12 — Performance: What Is the Machine Waiting For?

**Book alignment:** PyTorch From First Principles, Chapter 12

**Question this notebook isolates:** Does a warmed, repeated measurement with phase decomposition localize the step cost (forward vs backward vs optimizer) instead of a single naive wall-clock number? (CPU-only port of the chapter's CUDA method.)

In [ ]:
import time
import statistics
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())

## 1 — Benchmark hygiene: warmup plus median beats one naive timing

One wall-clock loop measures submission noise and outliers. Warmup plus repeated samples with median/IQR reports completed work robustly.

In [ ]:
def time_calls(fn, warmup=10, repeats=40):
    for _ in range(warmup):
        fn()
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - t0) * 1000)
    return samples

a = torch.randn(256, 256)
b = torch.randn(256, 256)
def workload():
    return (a @ b).sum()

t0 = time.perf_counter()
workload()
naive_ms = (time.perf_counter() - t0) * 1000
samples = time_calls(workload)
med = statistics.median(samples)
q = statistics.quantiles(samples, n=4)
iqr = q[2] - q[0]
print(f'naive one call: {naive_ms:.3f} ms')
print(f'warmed median: {med:.3f} ms  IQR: {iqr:.3f}  n={len(samples)}')
print(f'mean: {statistics.mean(samples):.3f} max: {max(samples):.3f}')

In [ ]:
assert len(samples) == 40
assert med > 0 and iqr >= 0
assert med < max(samples) + 1e-9
print('hygiene: warmed median/IQR verified')

## 2 — Localize before intervening: decompose the training step

Time forward-only, forward+backward, and the full step separately. The bottleneck phase — not the model in general — owns the intervention.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim=256, depth=4, classes=10):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(dim, dim), nn.GELU()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(dim, classes)
    def forward(self, x):
        return self.head(self.body(x))

torch.manual_seed(0)
mlp = MLP()
opt = torch.optim.AdamW(mlp.parameters(), lr=1e-3)
xb = torch.randn(64, 256)
yb = torch.randint(0, 10, (64,))

def fwd():
    return mlp(xb)

def fwd_bwd():
    opt.zero_grad()
    l = F.cross_entropy(mlp(xb), yb)
    l.backward()
    return l

def full_step():
    opt.zero_grad()
    l = F.cross_entropy(mlp(xb), yb)
    l.backward()
    opt.step()
    return l

m_fwd = statistics.median(time_calls(fwd))
m_fb = statistics.median(time_calls(fwd_bwd))
m_full = statistics.median(time_calls(full_step))
print(f'forward={m_fwd:.3f} ms  fwd+bwd={m_fb:.3f} ms  full={m_full:.3f} ms')
print(f'bwd share={(m_fb - m_fwd) / m_full * 100:.1f}%  opt share={(m_full - m_fb) / m_full * 100:.1f}%')

In [ ]:
assert m_fwd > 0 and m_fb >= m_fwd
assert m_full >= m_fb
print('phase decomposition verified')

## 3 — Batch sweep plus retained-graph growth

Latency, throughput, and live-allocation growth answer different questions. Keeping loss tensors alive retains graphs and grows memory; keeping floats does not.

In [ ]:
rows = []
for bs in [16, 64, 256]:
    xb2 = torch.randn(bs, 256)
    ms = statistics.median(time_calls(lambda: mlp(xb2), warmup=5, repeats=20))
    ex_s = bs / (ms / 1000)
    rows.append((bs, ms, ex_s))
    print(f'batch {bs:4d}  {ms:7.3f} ms/step  {ex_s:9.0f} ex/s')

mlp.eval()
kept_tensors, kept_floats = [], []
with torch.no_grad():
    ref = mlp(xb)
mlp.train()
for i in range(10):
    out = mlp(xb)
    kept_tensors.append(out.softmax(-1).amax(-1).mean())
    with torch.no_grad():
        kept_floats.append(float(mlp(xb).softmax(-1).amax(-1).mean()))
print('kept tensor grad_fn is None:', kept_tensors[0].grad_fn is None)
print('kept float type:', type(kept_floats[0]).__name__)
print('n kept:', len(kept_tensors), len(kept_floats))

In [ ]:
assert rows[2][1] >= rows[0][1]
assert rows[2][2] >= rows[0][2]
assert kept_tensors[0].grad_fn is not None
assert isinstance(kept_floats[0], float)
print('latency/throughput + retained-graph verified')

## What we earned

A number without a contract is not a measurement: warmup plus median/IQR defines the step time, phase brackets localize it, batch sweeps separate latency from throughput, and a climbing live set means retained graphs — not a leaking allocator.

Chapter 13 applies the same discipline to the compiler: correctness first, then what was captured, which guard failed, and whether reuse amortizes the cost.